# Experiment 1 (Part 2 of 2): Text Generation and Decoding Parameters with GPT-2 / Mistral

**Course:** HAAI302COM - Artificial Intelligence Laboratory
**Experiment 1:** Tokenization Analysis and Text Generation using Hugging Face Transformers

## Aim
To load a pre-trained causal language model (GPT-2 locally, Mistral-7B on a GPU), perform text generation, and study how decoding parameters (greedy, beam search, temperature, top-k, top-p, repetition penalty) change the output.

## Objectives
1. Load GPT-2 with `AutoModelForCausalLM` and inspect its configuration.
2. Run one forward pass by hand and read the next-token probability table.
3. Compare greedy decoding, beam search and sampling.
4. Visualise how `temperature`, `top_k` and `top_p` reshape the probability distribution.
5. Measure output quality with two simple numbers: **repetition** (distinct bigrams) and **perplexity**.
6. Run the same pipeline with Mistral-7B-Instruct and its chat template (Colab GPU section).

## Theory in one paragraph
A causal language model does one thing: given the tokens so far, it outputs a score (a **logit**) for every token in its vocabulary. Softmax turns those scores into probabilities. **Decoding** is the rule we use to pick the next token from that distribution: always pick the top one (greedy), keep several candidate sentences (beam search), or roll a weighted dice (sampling). The picked token is appended to the input and the process repeats. The model is the same in every case; only the picking rule changes, and that alone can turn a boring, repetitive answer into a creative one, or into nonsense.

## 0. Setup

- `AutoModelForCausalLM` loads a decoder-only model with its language-modelling head (the layer that produces one logit per vocabulary token).
- `torch.no_grad()` turns off gradient tracking because we only do inference.
- We fix the random seed with `torch.manual_seed` so sampling results are reproducible in class.

GPT-2 small (124M parameters) runs comfortably on a laptop CPU. The Mistral section at the end needs a GPU (Colab T4 or better).

In [1]:
# !pip install -q transformers torch pandas accelerate bitsandbytes
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"      # keep download bars out of the saved outputs
import transformers, huggingface_hub
transformers.logging.set_verbosity_error()            # hide informational library warnings
huggingface_hub.utils.logging.set_verbosity_error()
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

pd.set_option("display.max_colwidth", 120)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", device)

torch 2.14.0+cu130 | device: cpu


## 1. Load GPT-2 and look at what we loaded

We load the tokenizer and the model from the same checkpoint name. `model.config` holds the architecture numbers; `model.num_parameters()` counts the weights.

In [2]:
MODEL = "gpt2"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL).to(device).eval()   # eval() disables dropout
tok.pad_token = tok.eos_token          # GPT-2 has no pad token (see Part 1, Section 6.1)
tok.padding_side = "left"

In [3]:
cfg = model.config
pd.Series({
    "layers (n_layer)": cfg.n_layer,
    "attention heads (n_head)": cfg.n_head,
    "hidden size (n_embd)": cfg.n_embd,
    "vocabulary size": cfg.vocab_size,
    "max context (n_positions)": cfg.n_positions,
    "parameters (millions)": round(model.num_parameters() / 1e6, 1),
}).to_frame("GPT-2 small")

,GPT-2 small
layers (n_layer),12.0
attention heads (n_head),12.0
hidden size (n_embd),768.0
vocabulary size,50257.0
max context (n_positions),1024.0
parameters (millions),124.4


**Takeaway.** 12 transformer layers, 12 heads, 768-dimensional vectors, a vocabulary of 50,257 tokens and a context window of only 1,024 tokens. Every number you see later (e.g. the logits shape) comes from this table.

## 2. One forward pass by hand: from prompt to next-token probabilities

`generate()` hides a loop. Let us open the loop and do **one** step manually:
1. Tokenize the prompt.
2. Call `model(**inputs)`. It returns `logits` with shape `(batch, sequence_length, vocab_size)`: one score vector for **every position**.
3. Only the **last position** predicts the next token, so we take `logits[0, -1]`.
4. `softmax` turns scores into probabilities that sum to 1.

In [4]:
prompt = "The future of artificial intelligence is"
inputs = tok(prompt, return_tensors="pt").to(device)
with torch.no_grad():
    out = model(**inputs)
print("input_ids shape:", tuple(inputs["input_ids"].shape))
print("logits shape   :", tuple(out.logits.shape), "-> (batch, positions, vocabulary)")

input_ids shape: (1, 6)
logits shape   : (1, 6, 50257) -> (batch, positions, vocabulary)


In [5]:
last_logits = out.logits[0, -1]                    # scores for the token after "is"
probs = torch.softmax(last_logits, dim=-1)         # convert scores to probabilities
top = torch.topk(probs, 10)
next_table = pd.DataFrame({
    "token": [repr(tok.decode(i)) for i in top.indices],
    "logit": last_logits[top.indices].round(decimals=2).tolist(),
    "probability": top.values.round(decimals=4).tolist(),
})
next_table

,token,logit,probability
0,' uncertain',-114.900002,0.0766
1,' in',-115.000000,0.0694
2,' not',-115.459999,0.0441
3,' a',-115.540001,0.0407
4,' still',-115.699997,0.0345
5,' now',-116.059998,0.0240
6,' at',-116.129997,0.0224
7,' bright',-116.230003,0.0203
8,' going',-116.300003,0.0190
9,' unclear',-116.339996,0.0182


**Takeaway.**
- The model is not sure: the top token usually has well under 10% probability and many reasonable continuations follow it. This "spread" is exactly what the decoding parameters below act on.
- Every candidate starts with a space (`' a'`, `' in'`), because the prompt ended right after a word.
- Logits can be any real number; only after softmax do they become probabilities.

## 3. Greedy decoding, opened up step by step

**Greedy** = at every step, pick the single most probable token. We run six steps by hand and record what the model picked and how confident it was.

In [6]:
ids = inputs["input_ids"]
steps = []
for step in range(6):
    with torch.no_grad():
        p = torch.softmax(model(ids).logits[0, -1], dim=-1)
    best = int(p.argmax())                                         # greedy choice
    steps.append({"step": step + 1, "chosen token": repr(tok.decode(best)), "probability": round(float(p[best]), 4)})
    ids = torch.cat([ids, torch.tensor([[best]], device=device)], dim=1)   # append and repeat
pd.DataFrame(steps)

,step,chosen token,probability
0,1,' uncertain',0.0766
1,2,'.',0.3799
2,3,'\n',0.0756
3,4,'\n',0.9994
4,5,"'""'",0.0979
5,6,'We',0.1282


In [7]:
print(tok.decode(ids[0]))

The future of artificial intelligence is uncertain.

"We


**Takeaway.** Each step only looks one token ahead. The loop you just ran is exactly what `model.generate(do_sample=False)` does internally, just faster.

## 4. A helper for all experiments

`model.generate()` accepts every decoding parameter as a keyword argument. The ones we study:

| Parameter | What it controls | Too low | Too high |
|---|---|---|---|
| `max_new_tokens` | how many tokens to add | cut-off answers | slow, rambling |
| `do_sample` | `False` = deterministic, `True` = random sampling | | |
| `num_beams` | how many partial sentences to keep | 1 = greedy | slow, generic text |
| `temperature` | sharpens (<1) or flattens (>1) the distribution | repetitive, dull | incoherent |
| `top_k` | sample only from the k best tokens | 1 = greedy | rare junk tokens allowed |
| `top_p` | sample from the smallest set covering p of the probability | too narrow | junk allowed |
| `repetition_penalty` | lowers the score of tokens already used | loops | avoids even needed words |
| `no_repeat_ngram_size` | forbids repeating any n-gram | | can block names, legit phrases |

The helper below returns only the **new** text (without the prompt) so tables stay readable.

In [8]:
def generate(prompt, seed=42, **kwargs):
    torch.manual_seed(seed)                                    # same seed -> same random draws
    enc = tok(prompt, return_tensors="pt").to(device)
    kwargs.setdefault("max_new_tokens", 40)
    with torch.no_grad():
        out = model.generate(**enc, pad_token_id=tok.eos_token_id, **kwargs)
    return tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)

In [9]:
greedy_text = generate(prompt, do_sample=False, max_new_tokens=60)
print(prompt + greedy_text)

The future of artificial intelligence is uncertain.

"We're not sure what the future will look like," said Dr. Michael S. Schoenfeld, a professor of computer science at the University of California, Berkeley. "But we're not sure what the future will look like for artificial intelligence."

The future of


**Takeaway.** Within 60 tokens greedy decoding already **repeats itself**: "we're not sure what the future will look like" appears twice and the text heads back to "The future of". Once a phrase is in the context, repeating it becomes even more likely. Run it with `max_new_tokens=150` to watch it turn into a full loop. This is the classic failure that the other decoding strategies try to fix.

## 5. Beam search

Instead of keeping one sentence, **beam search** keeps the `num_beams` best partial sentences at every step and finally returns the one with the best total log-probability. With `return_dict_in_generate=True, output_scores=True` we can read the score of each returned beam.

In [10]:
enc = tok(prompt, return_tensors="pt").to(device)
with torch.no_grad():
    beams = model.generate(**enc, num_beams=5, num_return_sequences=3, max_new_tokens=30,
                           early_stopping=True, return_dict_in_generate=True, output_scores=True,
                           pad_token_id=tok.eos_token_id)
pd.DataFrame({
    "beam": [1, 2, 3],
    "score (avg log-prob)": beams.sequences_scores.round(decimals=3).tolist(),
    "continuation": [tok.decode(s[enc["input_ids"].shape[1]:], skip_special_tokens=True) for s in beams.sequences],
})

,beam,score (avg log-prob),continuation
0,1,-0.779,in the hands of the next generation of scientists and engineers.\n\nThe future of artificial intelligence is in the...
1,2,-0.842,"in the hands of the next generation of scientists and engineers.\n\n""The future of artificial intelligence is in th..."
2,3,-0.896,in the hands of the next generation of scientists and engineers.\n\nThe future of AI is in the hands of the next ge...


**Takeaway.** The three beams are very similar and still repetitive. Beam search maximises probability, and the most *probable* text is often the most *boring* text. Beam search works well for tasks with one right answer (translation, summarisation) but poorly for open-ended writing.

## 6. Temperature: reshaping the distribution

Sampling with temperature divides every logit by `T` before softmax:
- `T < 1` - gaps between scores grow, the top token dominates (more focused).
- `T = 1` - the model's own distribution.
- `T > 1` - gaps shrink, unlikely tokens gain probability (more random).

We take the same last-position logits from Section 2 and show the top-6 probabilities at several temperatures.

In [11]:
temps = [0.2, 0.7, 1.0, 1.5]
top6 = torch.topk(last_logits, 6).indices
temp_table = pd.DataFrame(
    {f"T={t}": torch.softmax(last_logits / t, dim=-1)[top6].round(decimals=3).tolist() for t in temps},
    index=[repr(tok.decode(i)) for i in top6],
)
temp_table

,T=0.2,T=0.7,T=1.0,T=1.5
' uncertain',0.572,0.170,0.077,0.021
' in',0.351,0.148,0.069,0.020
' not',0.036,0.077,0.044,0.015
' a',0.024,0.069,0.041,0.014
' still',0.011,0.055,0.034,0.013
' now',0.002,0.032,0.024,0.010


**Takeaway.** Read each column top to bottom. At `T=0.2` almost all probability sits on the first token (nearly greedy). At `T=1.5` the six tokens get similar shares and the rest of the 50,257-token vocabulary also gets more. The ranking never changes; only how *peaked* the distribution is.

In [12]:
rows = [{"temperature": t, "output": generate(prompt, do_sample=True, temperature=t, top_k=0)} for t in temps]
pd.DataFrame(rows)

,temperature,output
0,0.2,uncertain. But the future of artificial intelligence is not clear.\n\nThe future of artificial intelligence is unce...
1,0.7,"uncertain, but a new report in the journal Proceedings of the National Academy of Sciences shows that we can learn ..."
2,1.0,"solid and intelligent work is at least beginning."""
3,1.5,solid and intelligent work moves on in humanoid conversations abound: Google Vice President Gear VR weighed in we c...


**Takeaway.** Low temperature stays on topic but can still loop. Around 0.7 the text is usually most readable. At 1.5 (with no other filter, `top_k=0`) the model starts picking rare tokens and the text drifts into nonsense.

## 7. Top-k and top-p (nucleus) filtering

Temperature alone still lets every token in the vocabulary be picked occasionally. Filters cut the tail off:
- **top-k** keeps only the `k` highest-probability tokens.
- **top-p** sorts tokens by probability and keeps the smallest set whose probabilities add up to at least `p`. The set size adapts: when the model is confident, few tokens survive; when unsure, many survive.

Below we compute the cumulative probability and count how many tokens survive each `top_p` for our prompt.

In [13]:
sorted_p, sorted_idx = torch.sort(probs, descending=True)
cumulative = torch.cumsum(sorted_p, dim=0)
pd.DataFrame({
    "rank": range(1, 9),
    "token": [repr(tok.decode(i)) for i in sorted_idx[:8]],
    "probability": sorted_p[:8].round(decimals=4).tolist(),
    "cumulative": cumulative[:8].round(decimals=4).tolist(),
})

,rank,token,probability,cumulative
0,1,' uncertain',0.0766,0.0766
1,2,' in',0.0694,0.1460
2,3,' not',0.0441,0.1901
3,4,' a',0.0407,0.2307
4,5,' still',0.0345,0.2652
5,6,' now',0.0240,0.2892
6,7,' at',0.0224,0.3117
7,8,' bright',0.0203,0.3320


In [14]:
pd.DataFrame({
    "top_p": [0.5, 0.9, 0.95],
    "tokens kept (this prompt)": [int((cumulative < p).sum()) + 1 for p in [0.5, 0.9, 0.95]],
})

,top_p,tokens kept (this prompt)
0,0.50,21
1,0.90,361
2,0.95,794


**Takeaway.** After "is", the model is unsure, so even `top_p=0.5` keeps a large number of tokens and `0.95` keeps thousands. A fixed `top_k=50` would keep exactly 50 here and also exactly 50 after a prompt like "The capital of France is", where only one token makes sense. That adaptivity is why `top_p` is the more common default.

### 7.1 Confident prompt vs uncertain prompt

Let us check that claim with a prompt where the next token is nearly obvious.

In [15]:
def nucleus_size(text, p=0.9):
    with torch.no_grad():
        pr = torch.softmax(model(**tok(text, return_tensors="pt").to(device)).logits[0, -1], dim=-1)
    c = torch.cumsum(torch.sort(pr, descending=True).values, dim=0)
    return int((c < p).sum()) + 1, repr(tok.decode(pr.argmax()))

pd.DataFrame([(t, *nucleus_size(t)) for t in [prompt, "One, two, three, four,", "Monday, Tuesday, Wednesday,"]],
             columns=["prompt", "tokens in 90% nucleus", "most likely next"])

,prompt,tokens in 90% nucleus,most likely next
0,The future of artificial intelligence is,361,' uncertain'
1,"One, two, three, four,",19,' five'
2,"Monday, Tuesday, Wednesday,",3,' Thursday'


**Takeaway.** For the open-ended prompt the 90% nucleus holds hundreds of tokens; for the counting prompt about twenty; for the weekday prompt only a handful, because `' Thursday'` alone takes most of the probability. `top_p` automatically narrows when the model is confident and widens when it is not.

## 8. Repetition controls

Two parameters directly attack loops:
- `repetition_penalty` (e.g. 1.3) divides the logit of any token already in the text, making repeats less likely.
- `no_repeat_ngram_size` (e.g. 3) hard-bans any 3-token sequence from appearing twice.

We measure repetition with **distinct-2**: the number of unique word pairs divided by the total number of word pairs. 1.0 means no pair repeats; lower values mean more looping.

In [16]:
def distinct_2(text):
    words = text.split()
    pairs = list(zip(words, words[1:]))
    return round(len(set(pairs)) / max(len(pairs), 1), 3)

configs = {
    "greedy": dict(do_sample=False),
    "greedy + repetition_penalty=1.3": dict(do_sample=False, repetition_penalty=1.3),
    "greedy + no_repeat_ngram_size=3": dict(do_sample=False, no_repeat_ngram_size=3),
}
rep = pd.DataFrame([{"config": k, "output": (o := generate(prompt, max_new_tokens=60, **v)), "distinct-2": distinct_2(o)}
                    for k, v in configs.items()])
rep

,config,output,distinct-2
0,greedy,"uncertain.\n\n""We're not sure what the future will look like,"" said Dr. Michael S. Schoenfeld, a professor of compu...",0.854
1,greedy + repetition_penalty=1.3,"uncertain.\n""We're not sure what the next step will be,"" said Dr. Michael Siegel, a professor at Stanford Universit...",1.000
2,greedy + no_repeat_ngram_size=3,"uncertain.\n\n""We're not sure what the future will look like,"" said Dr. Michael S. Schoenfeld, a professor of compu...",0.953


**Takeaway.** Plain greedy has the lowest distinct-2 because of its loop. Both controls raise it, while staying deterministic (no sampling). `no_repeat_ngram_size` is a blunt tool: in a long answer it can stop the model from legitimately repeating a name like "New York City".

## 9. The full comparison: quality numbers for every strategy

We add one more number, **perplexity**: how surprised GPT-2 itself is by the generated text. We compute it by feeding prompt + output back into the model with `labels=input_ids`; the model returns the average loss, and `exp(loss)` is perplexity.
- Very **low** perplexity often means safe, repetitive text.
- Very **high** perplexity means the text contains tokens the model itself found unlikely (often nonsense).
- Good open-ended text sits in between and also has high distinct-2.

In [17]:
def perplexity(text):
    ids = tok(text, return_tensors="pt").input_ids.to(device)
    with torch.no_grad():
        loss = model(ids, labels=ids).loss            # average cross-entropy over tokens
    return round(float(torch.exp(loss)), 1)

In [18]:
strategies = {
    "greedy":               dict(do_sample=False),
    "beam (5)":             dict(num_beams=5, early_stopping=True),
    "sample T=0.7":         dict(do_sample=True, temperature=0.7, top_k=0),
    "sample T=1.5":         dict(do_sample=True, temperature=1.5, top_k=0),
    "top_k=50":             dict(do_sample=True, top_k=50),
    "top_p=0.9, T=0.8":     dict(do_sample=True, top_p=0.9, top_k=0, temperature=0.8),
    "top_p=0.9 + rep 1.2":  dict(do_sample=True, top_p=0.9, top_k=0, temperature=0.8, repetition_penalty=1.2),
}
results = []
for name, kw in strategies.items():
    o = generate(prompt, max_new_tokens=50, **kw)
    results.append({"strategy": name, "distinct-2": distinct_2(o), "perplexity": perplexity(prompt + o), "output": o})
results = pd.DataFrame(results)
results[["strategy", "distinct-2", "perplexity"]]

,strategy,distinct-2,perplexity
0,greedy,0.824,4.6
1,beam (5),0.395,2.3
2,sample T=0.7,1.000,5.5
3,sample T=1.5,1.000,4340.7
4,top_k=50,0.905,11.1
5,"top_p=0.9, T=0.8",0.930,8.1
6,top_p=0.9 + rep 1.2,1.000,21.2


In [19]:
results[["strategy", "output"]]

,strategy,output
0,greedy,"uncertain.\n\n""We're not sure what the future will look like,"" said Dr. Michael S. Schoenfeld, a professor of compu..."
1,beam (5),in the hands of the next generation of scientists and engineers.\n\nThe future of artificial intelligence is in the...
2,sample T=0.7,"uncertain, but a new report in the journal Proceedings of the National Academy of Sciences shows that we can learn ..."
3,sample T=1.5,solid and intelligent work moves on in humanoid conversations abound: Google Vice President Gear VR weighed in we c...
4,top_k=50,uncertain and could ultimately create new problems for the U.S. economy and may even bring us down. The U.S. Govern...
5,"top_p=0.9, T=0.8","uncertain and could be even more uncertain than the future of science.\n\nFor example, we can't predict if a person..."
6,top_p=0.9 + rep 1.2,"uncertain and could be even more important than the amount or quality that we currently understand,"" said Simon Har..."


**Takeaway.** Read the two tables together:
- **beam** has the lowest perplexity of all and by far the lowest distinct-2: the most "probable" text is a loop. **greedy** is similar but slightly less extreme.
- **T=1.5** with no filter has a perplexity in the thousands: every pair of words is new, but the text is nonsense.
- **moderate temperature (0.7 to 0.8), with or without top-p**, lands in the sweet spot: distinct-2 near 1.0 with a perplexity that is only a little above greedy. Adding a mild repetition penalty raises perplexity a bit because it forces less likely word choices. This is why most chat APIs default to something like `temperature=0.7, top_p=0.9`.

Your exact numbers depend on the seed; run the cell with `seed=1`, `seed=7` to see the variation.

## 10. The model's default generation settings

Every Hub model ships a `generation_config` that `generate()` uses when you pass nothing. It is worth checking, because defaults differ between models.

In [20]:
gc = model.generation_config.to_dict()
explicit = {k: v for k, v in gc.items() if v is not None and not k.startswith("_")}   # only values the model sets
print("explicitly set by GPT-2:", explicit)
print("unset (None) settings:", sum(v is None for v in gc.values()))

explicitly set by GPT-2: {'bos_token_id': 50256, 'eos_token_id': 50256, 'transformers_version': '5.16.1'}
unset (None) settings: 69


**Takeaway.** GPT-2 sets only its special token IDs (50256 = `<|endoftext|>`). Every other setting is `None`, which means `generate()` falls back to its built-in defaults: greedy decoding (no sampling, one beam) and a short default length. That is why we always pass `max_new_tokens` and `do_sample` explicitly. Newer chat models usually ship their own recommended `temperature` and `top_p` here.

## 11. Batched generation (left padding in action)

To serve many prompts at once we tokenize them together. Because we set `padding_side="left"` in Section 1, shorter prompts get padding at the start, so all prompts end at the same position and generation continues right after real text.

In [21]:
batch_prompts = ["Large language models are", "In Pune, the weather today is", "Python is popular because"]
enc = tok(batch_prompts, return_tensors="pt", padding=True).to(device)
with torch.no_grad():
    out = model.generate(**enc, max_new_tokens=20, do_sample=True, top_p=0.9, temperature=0.8, pad_token_id=tok.eos_token_id)
pd.DataFrame({"prompt": batch_prompts,
              "padded input_ids": enc["input_ids"].tolist(),
              "continuation": tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)})

,prompt,padded input_ids,continuation
0,Large language models are,"[50256, 50256, 50256, 50256, 21968, 3303, 4981, 389]",designed to use only a handful of languages. They are not intended to be the equivalent of a full
1,"In Pune, the weather today is","[818, 350, 1726, 11, 262, 6193, 1909, 318]",very mild. We are seeing more snowfall in the day with some showers in the evening. The
2,Python is popular because,"[50256, 50256, 50256, 50256, 37906, 318, 2968, 780]",it is easy to use. You can use it to create your own plugins and tools for your projects


**Takeaway.** The shorter prompts begin with one or more `50256` padding IDs, and the attention mask hides them. Try setting `tok.padding_side = "right"` and rerunning: the model would then generate *after the padding* and the outputs degrade (transformers also prints a warning).

## 12. Mistral-7B-Instruct on a GPU (Colab)

GPT-2 is a 2019 base model: it only continues text. Mistral-7B-Instruct is a modern **instruction-tuned** model, so it expects a chat format. The tokenizer stores that format as a **chat template**; `apply_chat_template` turns a list of messages into the exact string the model was trained on.

The template step needs only the tokenizer, so it runs anywhere. Loading the 7B model needs a GPU: on Colab choose *Runtime > Change runtime type > T4 GPU*. We load it in **4-bit** (`BitsAndBytesConfig(load_in_4bit=True)`) so it fits in about 5 GB of GPU memory.

In [22]:
MISTRAL = "mistralai/Mistral-7B-Instruct-v0.3"
m_tok = AutoTokenizer.from_pretrained(MISTRAL)
messages = [{"role": "user", "content": "Explain top-p sampling to a first-year student in two sentences."}]
chat_text = m_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(repr(chat_text))
print("tokens:", m_tok.tokenize(chat_text))

'<s>[INST] Explain top-p sampling to a first-year student in two sentences.[/INST]'
tokens: ['<s>', '[INST]', '▁Expl', 'ain', '▁top', '-', 'p', '▁sampling', '▁to', '▁a', '▁first', '-', 'year', '▁student', '▁in', '▁two', '▁sentences', '.', '[/INST]']


**Takeaway.** The user message is wrapped in `[INST] ... [/INST]` with a beginning-of-sequence token `<s>`. If you skip the template and send plain text, an instruction-tuned model behaves noticeably worse, because it never saw raw text during instruction tuning.

In [23]:
if torch.cuda.is_available():
    from transformers import BitsAndBytesConfig
    m_model = AutoModelForCausalLM.from_pretrained(
        MISTRAL, device_map="auto",
        quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16))
    print("parameters (billions):", round(m_model.num_parameters() / 1e9, 2))
else:
    m_model = None
    print("No GPU found: skipping Mistral-7B. Run this section on Colab with a T4 GPU.")

No GPU found: skipping Mistral-7B. Run this section on Colab with a T4 GPU.


In [24]:
if m_model is not None:
    m_inputs = m_tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(m_model.device)
    for name, kw in {"greedy": dict(do_sample=False),
                     "top_p=0.9, T=0.7": dict(do_sample=True, top_p=0.9, temperature=0.7)}.items():
        torch.manual_seed(42)
        out = m_model.generate(**m_inputs, max_new_tokens=80, pad_token_id=m_tok.eos_token_id, **kw)
        print(f"--- {name}\n" + m_tok.decode(out[0, m_inputs["input_ids"].shape[1]:], skip_special_tokens=True))

**Takeaway (when run on GPU).** An instruction-tuned 7B model rarely loops even with greedy decoding, because instruction tuning taught it to produce a complete answer and then emit the end-of-sequence token. Decoding parameters still change the wording, length and variety of the answer.

## 13. Results summary

| Strategy | distinct-2 | perplexity | Your comment on the output |
|---|---|---|---|
| greedy | | | |
| beam (5) | | | |
| sample T=1.5 | | | |
| top_p=0.9, T=0.8 | | | |

## Conclusion
The language model produces the same probability distribution regardless of the decoding method. Greedy and beam search choose the highest-probability path and tend to produce repetitive text. Sampling with temperature adds variety, and top-k / top-p filtering removes the unreliable tail of the distribution. A moderate temperature with nucleus sampling and a mild repetition penalty gives the best balance between coherence (perplexity) and diversity (distinct-2).

## Viva questions
1. What is the shape of `logits` returned by a causal LM, and which position do we use to predict the next token?
2. Why does greedy decoding tend to repeat itself?
3. What does dividing logits by a temperature of 0.2 do to the probability distribution?
4. Explain why `top_p` adapts to the model's confidence while `top_k` does not.
5. Why does low perplexity not always mean better generated text?
6. What is a chat template, and why do instruction-tuned models need it?

## Exercise
Pick a task of your choice (a product description, a poem, a factual answer). For each strategy in Section 9, generate an answer with three different seeds and decide which strategy you would use in production for that task. Justify your choice with the distinct-2 and perplexity numbers.